In [49]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/fall-2026-csulb-pr-in-class-competiton/sample-submission.csv
/kaggle/input/competitions/fall-2026-csulb-pr-in-class-competiton/submission.csv
/kaggle/input/competitions/fall-2026-csulb-pr-in-class-competiton/Train.csv
/kaggle/input/competitions/fall-2026-csulb-pr-in-class-competiton/Test.csv


# **Plots - Model Training**

In [50]:
#plot f1 scores
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import clear_output


def plot_class_f1_bar(cls_f1, class_names, title="Per-Class F1 Scores", in_place=True, ax=None):
    """Plots your exact per-class F1 bar chart with green Macro F1 line onto ax (or standalone)."""
    standalone = ax is None
    if standalone:
        if in_place:
            clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6.5, 2.3))

    colors = ["crimson" if str(c) in ["3", "4", "6"] else "steelblue" for c in class_names]
    bars = ax.bar(class_names, cls_f1, color=colors, edgecolor="black", alpha=0.85, width=0.65)

    macro_f1 = float(np.mean(cls_f1))
    ax.axhline(macro_f1, color="darkgreen", linestyle="--", linewidth=1.2, label=f"Macro F1: {macro_f1:.4f}")
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("F1", fontsize=8)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.tick_params(axis="both", labelsize=7.5)
    ax.legend(loc="upper right", fontsize=7.5)
    ax.grid(axis="y", alpha=0.3)

    for bar, score in zip(bars, cls_f1):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            score + 0.03,
            f"{score:.2f}",
            ha="center",
            va="bottom",
            fontsize=6.5,
            fontweight="bold",
        )

    if standalone:
        plt.tight_layout()
        if in_place:
            display(fig)
            plt.close(fig)
        else:
            plt.show()



In [51]:
#plot resnet neuron graph

import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection


def plot_resnet_neuron_graph(model, feat_names, class_names, top_percentile=88, ax=None):
    """Compact, vectorized neural connection graph using Matplotlib LineCollection."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 3.6))

    # 1. Extract weight matrices from in_proj, all loop_cell linear layers, and head
    w_in = [m for m in model.in_proj if isinstance(m, nn.Linear)][0].weight.detach().cpu().numpy()
    w_loops = [m.weight.detach().cpu().numpy() for m in model.loop_cell if isinstance(m, nn.Linear)]
    w_out = (model.head if isinstance(model.head, nn.Linear) else
             [m for m in model.head if isinstance(m, nn.Linear)][-1]).weight.detach().cpu().numpy()

    n_h = w_in.shape[0]
    # Merge skip-connection half [h, x0] -> [h] if first loop layer takes concatenated input
    if w_loops[0].shape[1] == 2 * n_h:
        w0 = w_loops[0]
        w_loops[0] = 0.5 * (np.abs(w0[:, :n_h]) + np.abs(w0[:, n_h:])) * np.sign(w0[:, :n_h] + w0[:, n_h:])

    # 2. Compute Y-coordinates for every neuron column (Input -> Hidden Layers -> Output)
    weights = [w_in] + w_loops + [w_out]
    col_sizes = [w.shape[1] for w in weights] + [w_out.shape[0]]
    ys = [np.linspace(0, -n_h + 1, sz) for sz in col_sizes]

    # 3. Vectorized edge creation: filter top weights and build line segments + colors
    segs, colors, widths = [], [], []
    for x, (W, y_src, y_dst) in enumerate(zip(weights, ys[:-1], ys[1:])):
        dst_idx, src_idx = np.where(np.abs(W) >= np.percentile(np.abs(W), top_percentile))
        vals = W[dst_idx, src_idx]
        segs.extend([[(x, y_src[s]), (x + 1, y_dst[d])] for s, d in zip(src_idx, dst_idx)])
        colors.extend(["crimson" if v > 0 else "royalblue" for v in vals])
        widths.extend(np.clip(np.abs(vals) * 1.1, 0.3, 1.8))

    ax.add_collection(LineCollection(segs, colors=colors, linewidths=widths, alpha=0.45))

    # 4. Plot neuron dots, labels, and the orange recurrent loop arrow
    for x, y_col in enumerate(ys):
        ax.scatter(np.full_like(y_col, x), y_col, s=30, c="lightgray", edgecolors="black", zorder=3)

    for y, name in zip(ys[0], feat_names):
        ax.text(-0.08, y, name, fontsize=6.5, ha="right", va="center")
    for y, cname in zip(ys[-1], class_names):
        is_rare = str(cname) in ["3", "4", "6"]
        ax.text(len(weights) + 0.08, y, f"Class {cname}", fontsize=6.5, ha="left", va="center",
                color="crimson" if is_rare else "black", fontweight="bold" if is_rare else "normal")

    if len(w_loops) >= 2:
        ax.add_patch(patches.FancyArrowPatch(
            (len(weights) - 1, 0.3), (2.0, 0.3), connectionstyle="arc3,rad=-0.25",
            arrowstyle="->,head_length=5,head_width=3", color="darkorange", lw=1.5
        ))

    ax.set_xlim(-1.2, len(weights) + 0.95)
    ax.set_ylim(-n_h - 0.5, 1.5)
    ax.axis("off")
    ax.set_title(f"Live Weights ({len(w_loops)}-Layer Loop | Red=+, Blue=-)", fontsize=9.5, fontweight="bold")




In [52]:
#plot_generalization_calib_regularization_trends, plot_RareClassProb_Overconfidence_Weight_Norm, plot_perClass_pr_auc, plot_confusion_matrix

def plot_generalization_calib_regularization_trends(history: dict, ax):
    """Plots Train/Val Loss alongside the Va/Tr Overfit Ratio on a dual-axis panel."""
    eps = history["ep"]

    ax.plot(eps, history["tr_loss"], label="Train Loss", color="#1f77b4", lw=1.8)
    ax.plot(eps, history["va_loss"], label="Val Loss", color="#ff7f0e", lw=1.8)
    ax.set_ylabel("Loss")
    ax.set_xlabel("Epoch")
    ax.set_title(
        f"Loss & Overfit Ratio ({history['loss_ratio'][-1]:.2f}x)", fontsize=10, fontweight="bold"
    )
    ax.grid(True, alpha=0.25)

    ax_ratio = ax.twinx()
    ax_ratio.plot(eps, history["loss_ratio"], label="Va/Tr Ratio", color="#d62728", ls="--", alpha=0.75)
    ax_ratio.axhline(1.0, color="gray", ls=":", lw=1)
    ax_ratio.set_ylabel("Va/Tr Ratio", color="#d62728")
    ax_ratio.tick_params(axis="y", labelcolor="#d62728")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_ratio.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)


def plot_RareClassProb_Overconfidence_Weight_Norm(history: dict, ax):
    """Plots Rare-Class Probability (3,4,6), Overconfidence Gap, and L2 Weight Norm."""
    eps = history["ep"]

    ax.plot(eps, history["rare_true_p"], label="RareP (3,4,6)", color="#2ca02c", lw=1.8)
    ax.plot(eps, history["overconf_gap"], label="ConfGap (Conf-Acc)", color="#9467bd", lw=1.8)
    ax.axhline(0.0, color="gray", ls=":", lw=1)
    ax.set_ylabel("Probability / Gap")
    ax.set_xlabel("Epoch")
    ax.set_title(
        f"RareP: {history['rare_true_p'][-1]:.3f} | "
        f"ConfGap: {history['overconf_gap'][-1]:+.3f} | "
        f"W_norm: {history['w_norm'][-1]:.1f}",
        fontsize=10,
        fontweight="bold",
    )
    ax.grid(True, alpha=0.25)

    ax_wn = ax.twinx()
    ax_wn.plot(eps, history["w_norm"], label="W_norm (L2)", color="#8c564b", ls="--", alpha=0.75)
    ax_wn.set_ylabel("Weight Norm", color="#8c564b")
    ax_wn.tick_params(axis="y", labelcolor="#8c564b")

    lines3, labels3 = ax.get_legend_handles_labels()
    lines4, labels4 = ax_wn.get_legend_handles_labels()
    ax.legend(lines3 + lines4, labels3 + labels4, loc="upper left", fontsize=8)


def plot_perClass_pr_auc(y_true: np.ndarray, va_probs: np.ndarray, class_names, ax):
    """Plots horizontal per-class PR-AUC (Average Precision) bars highlighting rare classes."""
    num_classes = len(class_names)
    ap_scores = []
    for c in range(num_classes):
        y_bin = (y_true == c).astype(int)
        ap = average_precision_score(y_bin, va_probs[:, c]) if y_bin.sum() > 0 else 0.0
        ap_scores.append(ap)

    macro_ap = float(np.mean(ap_scores))
    y_pos = np.arange(num_classes)
    colors = ["#d62728" if str(lbl) in {"3", "4", "6"} else "#1f77b4" for lbl in class_names]

    bars = ax.barh(y_pos, ap_scores, color=colors, alpha=0.85, edgecolor="black", linewidth=0.5)
    ax.axvline(macro_ap, color="black", linestyle="--", linewidth=1.2, label=f"Macro PR-AUC: {macro_ap:.3f}")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([str(c) for c in class_names], fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlim(0.0, 1.08)
    ax.set_xlabel("PR-AUC (Average Precision)")
    ax.set_title(f"Per-Class PR-AUC (Macro: {macro_ap:.3f})", fontsize=10, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

    for bar, score in zip(bars, ap_scores):
        ax.text(score + 0.015, bar.get_y() + bar.get_height() / 2, f"{score:.2f}", va="center", fontsize=7.5)


def plot_confusion_matrix(y_true: np.ndarray, va_probs: np.ndarray, class_names, ax):
    """Plots row-normalized recall heatmap annotated with raw counts inside the dashboard."""
    num_classes = len(class_names)
    y_pred = np.argmax(va_probs, axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, np.maximum(row_sums, 1), dtype=float)

    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_title("Validation Confusion Matrix (Count / Row Recall)", fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")

    tick_marks = np.arange(num_classes)
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels([str(c) for c in class_names], rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels([str(c) for c in class_names], fontsize=8)

    for i in range(num_classes):
        for j in range(num_classes):
            count = cm[i, j]
            if count > 0:
                text_color = "white" if cm_norm[i, j] > 0.5 else "black"
                ax.text(j, i, f"{count}", ha="center", va="center", color=text_color, fontsize=7.5)


In [53]:
import matplotlib.pyplot as plt


def plot_dashboard_legend():
    """Renders a compact 3x2 Matplotlib reference card matching the live fold dashboard layout."""
    panels = [
        [
            (
                "[1] Per-Class F1 Bar Chart",
                "• Raw argmax F1 per class vs. Macro F1 dashed line.\n"
                "• Red bars highlight ultra-rare classes (3, 4, 6).",
                "#1f77b4",
            ),
            (
                "[2] Live Weights (Loop ResNet Graph)",
                "• Top-percentile weights (Red = positive, Blue = negative).\n"
                "• Shows which input features drive hidden & output class nodes.",
                "#d62728",
            ),
        ],
        [
            (
                "[3] Loss & Overfit Ratio (Va/Tr)",
                "• Train vs. Validation loss curves + Va/Tr ratio (dashed red).\n"
                "• Ratio > 1.0 signals overfitting; < 1.0 is healthy under noise/sampling.",
                "#ff7f0e",
            ),
            (
                "[4] RareP, Overconfidence & Weight Norm",
                "• RareP: mean predicted prob on true rare rows (higher = better).\n"
                "• ConfGap: Conf − Acc (near 0 = calibrated) | W_norm: L2 weight size.",
                "#2ca02c",
            ),
        ],
        [
            (
                "[5] Per-Class PR-AUC (Ranking Quality)",
                "• Threshold-free ranking score (0 to 1) per class.\n"
                "• High PR-AUC + low F1 means post-training multiplier tuning can fix it.",
                "#9467bd",
            ),
            (
                "[6] Validation Confusion Matrix",
                "• Rows = True Class, Columns = Predicted Class (annotated with counts).\n"
                "• Dark diagonal = high recall; off-diagonal cells = class bleed.",
                "#17becf",
            ),
        ],
    ]

    fig, axes = plt.subplots(
        3, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1.0, 1.55]}
    )
    fig.suptitle(
        "Live Fold Dashboard Guide (3×2 Panel Layout)",
        fontsize=11,
        fontweight="bold",
        y=0.99,
    )

    for r in range(3):
        for c in range(2):
            ax = axes[r, c]
            title, desc, accent = panels[r][c]
            ax.set_facecolor("#f8f9fa")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("#d0d7de")
                spine.set_linewidth(1.0)

            ax.text(
                0.025, 0.78, title, transform=ax.transAxes,
                fontsize=9.5, fontweight="bold", color=accent, va="top"
            )
            ax.text(
                0.025, 0.48, desc, transform=ax.transAxes,
                fontsize=8.5, color="#24292f", va="top", linespacing=1.45
            )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    plt.close(fig)


# Alias so existing calls work without renaming
print_dashboard_legend = plot_dashboard_legend


In [54]:
#stop reason

def print_stop_reason(
    ep: int,
    max_epochs: int,
    stale: int,
    patience: int,
    best_ep: int,
    best_sel: float,
    best_stats: dict,
):
    """Prints a concise summary explaining why fold training terminated."""
    if stale >= patience and ep < max_epochs:
        reason = f"Early Stopping — no improvement for {stale} epochs (patience={patience})"
    else:
        reason = f"Max Epochs Reached ({ep}/{max_epochs})"

    print(
        f"Stop Reason : {reason}\n"
        f"Best Ckpt   : Ep {best_ep} | Sel: {best_sel:.4f} | "
        f"MacroF1: {best_stats['f1']:.4f} | Bot4F1: {best_stats['bot4_f1']:.4f} | "
        f"PR-AUC: {best_stats['pr_auc']:.4f}"
    )

In [55]:
#depreciated print_pr_auc_report
from sklearn.metrics import average_precision_score, precision_recall_curve, auc

def print_pr_auc_report(oof_probs: np.ndarray, y: np.ndarray, class_names: np.ndarray):
    """Prints per-class and macro PR-AUC (Average Precision & Trapezoidal PR-AUC)."""
    num_classes = len(class_names)
    ap_scores = []
    trapz_scores = []

    print(f"\n{'Class':<22} | {'Support':>7} | {'Avg Prec (PR-AUC)':>17} | {'Trapz PR-AUC':>12}")
    print("-" * 68)

    for c in range(num_classes):
        y_true_bin = (y == c).astype(int)
        support = int(y_true_bin.sum())
        probs_c = oof_probs[:, c]

        # 1. Average Precision (standard step-wise PR-AUC, preferred for rare classes)
        ap = average_precision_score(y_true_bin, probs_c) if support > 0 else 0.0

        # 2. Trapezoidal PR-AUC (area under interpolated precision-recall curve)
        prec, rec, _ = precision_recall_curve(y_true_bin, probs_c)
        trapz_auc = auc(rec, prec) if support > 0 else 0.0

        ap_scores.append(ap)
        trapz_scores.append(trapz_auc)

        print(f"{str(class_names[c]):<22} | {support:>7d} | {ap:>17.4f} | {trapz_auc:>12.4f}")

    print("-" * 68)
    print(f"{'MACRO AVERAGE':<22} | {len(y):>7d} | {np.mean(ap_scores):>17.4f} | {np.mean(trapz_scores):>12.4f}\n")


In [56]:
#plot live fold dashboard (sub function calling version)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, confusion_matrix


def plot_live_fold_dashboard(
    cls_f1,
    class_names,
    model,
    feat_names,
    title,
    top_percentile,
    history=None,
    handle=None,
    y_true=None,
    va_probs=None,
    show_dashboard: bool = True,
):
    """Creates/updates a 3x2 fold dashboard, or prints a compact per-class PR-AUC line if show_dashboard=False."""
    has_eval = y_true is not None and va_probs is not None

    # Compact single-line text mode when plots are disabled
    if not show_dashboard:
        if has_eval:
            # Sort classes numerically (0..10) and format as 2-digit percentages
            order = sorted(
                range(len(class_names)),
                key=lambda c: int(class_names[c]) if str(class_names[c]).isdigit() else c
            )
            vals = [
                f"{int(round(average_precision_score((y_true == c), va_probs[:, c]) * 100)):02d}"
                if (y_true == c).sum() > 0 else "00"
                for c in order
            ]
            # Shorten title: "Fold 5 Ep 280 | MacroF1: 0.370 | Bot4F1: 0.022 | PR-AUC: 0.400 | Tr/Va: 0.824/0.710 (0.86x)"
            short_title = (
                title.replace("Fold ", "F")
                .replace(" Ep ", "E")
                .replace("MacroF1: ", "mF1:")
                .replace("Bot4F1: ", "b4:")
                .replace("PR-AUC: ", "PR:")
                .replace("Tr/Va: ", "L:")
                .replace(" | ", " ")
                .replace("0.", ".")
                .split(" (")[0]  # drops "(0.86x)"
            )
            print(f"{short_title} AP[{' '.join(vals)}]")
        else:
            print(title)
        return handle



    has_hist = history is not None and len(history.get("ep", [])) > 0

    if has_hist or has_eval:
        fig = plt.figure(figsize=(15, 10.2))
        gs = fig.add_gridspec(3, 2, width_ratios=[1.0, 1.55], height_ratios=[1.1, 1.0, 1.05])
        ax_bar = fig.add_subplot(gs[0, 0])
        ax_net = fig.add_subplot(gs[0, 1])
        ax_loss = fig.add_subplot(gs[1, 0])
        ax_cal = fig.add_subplot(gs[1, 1])
        ax_pr = fig.add_subplot(gs[2, 0])
        ax_cm = fig.add_subplot(gs[2, 1])
    else:
        fig, (ax_bar, ax_net) = plt.subplots(
            1, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1.0, 1.55]}
        )

    # Row 1: Existing Class F1 & Neuron Graph
    plot_class_f1_bar(cls_f1, class_names, title=title, ax=ax_bar)
    plot_resnet_neuron_graph(model, feat_names, class_names, top_percentile, ax=ax_net)

    # Row 2: Generalization, Calibration & Regularization Trends
    if has_hist:
        plot_generalization_calib_regularization_trends(history, ax=ax_loss)
        plot_RareClassProb_Overconfidence_Weight_Norm(history, ax=ax_cal)
    elif has_eval:
        ax_loss.axis("off")
        ax_cal.axis("off")

    # Row 3: Per-Class PR-AUC & Confusion Matrix
    if has_eval:
        plot_perClass_pr_auc(y_true, va_probs, class_names, ax=ax_pr)
        plot_confusion_matrix(y_true, va_probs, class_names, ax=ax_cm)
    elif has_hist:
        ax_pr.axis("off")
        ax_cm.axis("off")

    plt.tight_layout()
    if handle is None:
        handle = display(fig, display_id=True)
    else:
        handle.update(fig)
    plt.close(fig)
    return handle



# **Feature Engineering**

In [57]:
#plot_rare_distributions

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_rare_distributions(X_df: pd.DataFrame, y_raw: np.ndarray, rare_codes=("3", "4", "6")):
    """Plots zoomed, outlier-resistant distributions of rare classes (3, 4, 6) vs others."""
    y_str = pd.Series(y_raw).astype(str).values
    is_other = ~np.isin(y_str, rare_codes)
    cols = [c for c in X_df.columns if c not in ["patient_ID", "Rhythm"]]

    # Robust MAD-based separation score (immune to extreme sensor outliers)
    sep_scores = {}
    for col in cols:
        other_vals = X_df.loc[is_other, col].values
        mad = max(np.median(np.abs(other_vals - np.median(other_vals))) * 1.4826, 1e-4)
        sep_scores[col] = max(
            abs(np.median(X_df.loc[y_str == code, col].values) - np.median(other_vals)) / mad
            for code in rare_codes if np.any(y_str == code)
        )

    sorted_cols = sorted(cols, key=lambda c: sep_scores[c], reverse=True)
    n_cols = 4
    n_rows = int(np.ceil(len(sorted_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.0 * n_rows))
    axes = axes.flatten()
    palette = {"3": "crimson", "4": "darkorange", "6": "royalblue"}

    for idx, col in enumerate(sorted_cols):
        ax = axes[idx]
        all_vals = X_df[col].values
        # Zoom x-axis to [0.5%, 99.5%] quantiles so outliers never squash av_log_ratio or QRSDuration
        lo, hi = np.quantile(all_vals, [0.005, 0.995])
        if hi - lo < 1e-4:
            lo, hi = float(np.min(all_vals)), float(np.max(all_vals)) + 1e-3
        bins = np.linspace(lo, hi, 28)

        ax.hist(np.clip(X_df.loc[is_other, col], lo, hi), bins=bins, alpha=0.35, color="slategray", label="Others", log=True)
        for code in rare_codes:
            vals = X_df.loc[y_str == code, col].values
            if len(vals) > 0:
                ax.hist(np.clip(vals, lo, hi), bins=bins, alpha=0.75, color=palette[code], label=f"Class {code}", log=True)
                ax.axvline(np.clip(np.median(vals), lo, hi), color=palette[code], linestyle="--", linewidth=1.4)

        ax.set_xlim(lo, hi)
        ax.set_ylim(0.5, len(all_vals) * 1.3)
        ax.set_title(f"{col}\n(Robust Shift: {sep_scores[col]:.2f}σ)", fontsize=9, fontweight="bold")
        ax.tick_params(labelsize=7.5)
        if idx == 0:
            ax.legend(fontsize=7.5, loc="upper right")

    for j in range(len(sorted_cols), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Outlier-Zoomed Rare Classes (3, 4, 6) vs. Others (Log Count Scale)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()
    return pd.Series(sep_scores).sort_values(ascending=False)



In [58]:
def plot_indicator_grid(X_df: pd.DataFrame, y_raw: np.ndarray):
    """Plots a zoomed 1x3 distribution grid specifically for the 3 rare-class indicators."""
    y_str = pd.Series(y_raw).astype(str).values
    is_other = ~np.isin(y_str, ["3", "4", "6"])

    specs = [
        ("ind_class3_avnrt", "3", "crimson", "Class 3 (AVNRT) Indicator"),
        ("ind_class4_avrt", "4", "darkorange", "Class 4 (AVRT) Indicator"),
        ("ind_class6_saawr", "6", "royalblue", "Class 6 (SAAWR) Indicator"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    bins = np.linspace(0.0, 1.0, 26)

    for ax, (col, target_code, color, title) in zip(axes, specs):
        other_vals = X_df.loc[is_other, col].values
        target_vals = X_df.loc[y_str == target_code, col].values

        # Plot counts on log-scale y-axis so non-zero tail separation is clearly visible
        ax.hist(other_vals, bins=bins, alpha=0.40, color="slategray", label=f"Others (n={len(other_vals)})", log=True)
        for code, c_col in [("3", "crimson"), ("4", "darkorange"), ("6", "royalblue")]:
            vals = X_df.loc[y_str == code, col].values
            ax.hist(vals, bins=bins, alpha=0.75, color=c_col, label=f"Class {code} (n={len(vals)})", log=True)

        ax.axvline(np.median(target_vals), color=color, linestyle="--", linewidth=1.8,
                   label=f"Class {target_code} Median: {np.median(target_vals):.2f}")
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.5, len(other_vals) * 1.5)
        ax.set_xlabel("Indicator Affinity Score [0, 1]", fontsize=8.5)
        ax.set_ylabel("Patient Count (Log Scale)", fontsize=8.5)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7.5, loc="upper right")

    plt.tight_layout()
    plt.show()

In [59]:
#add_rare_indicator_features (features to seperate rare classes ind_class3_avnrt ind_class4_avrt ind_class6_saawr)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def add_rare_indicator_features(df: pd.DataFrame) -> pd.DataFrame:
    """Separates Class 3 (145-188 BPM), Class 4 (>192 BPM & short ST-T), and Class 6 (young resting)."""
    vr = df["VentricularRate"].clip(20, 300)
    ar = df["AtrialRate"].clip(20, 450)
    qrs = df["QRSDuration"].clip(40, 250)
    age = df["PatientAge"].clip(1, 110)
    stt = (df["TOffset"] - df["QOffset"]).clip(1)

    t_rad = np.deg2rad(df["TAxis"]) if "TAxis" in df.columns else np.arctan2(df["t_sin"], df["t_cos"])
    t_sin = np.sin(t_rad)

    av_lock = np.exp(-0.5 * (np.log(ar / vr) / 0.15) ** 2)
    narrow_qrs = 1.0 / (1.0 + np.exp((qrs - 110.0) / 8.0))
    irreg = np.abs(df["QRSCount"] - (vr / 60.0) * 10.0) if "QRSCount" in df.columns else 0.0
    regular_rhythm = np.exp(-0.5 * (irreg / 1.2) ** 2)

    # 1. Class 3 (AVNRT): regular narrow tachycardia centered at 165 BPM (below 192 BPM)
    rate_3 = np.exp(-0.5 * ((vr - 165.0) / 16.0) ** 2)
    raw_3 = av_lock * narrow_qrs * regular_rhythm * rate_3

    # 2. Class 4 (AVRT): extreme tachycardia (>192 BPM) + compressed ST-T span (<88)
    rate_4 = 1.0 / (1.0 + np.exp(-(vr - 192.0) / 10.0))
    short_stt_4 = 1.0 / (1.0 + np.exp((stt - 88.0) / 8.0))
    raw_4 = av_lock * narrow_qrs * regular_rhythm * rate_4 * short_stt_4

    # 3. Class 6 (SAAWR): young age (<35) + resting rate (55-85 BPM) + upright T-wave (t_sin > 0.5)
    rate_6 = np.exp(-0.5 * ((vr - 70.0) / 14.0) ** 2)
    age_6 = 1.0 / (1.0 + np.exp((age - 34.0) / 8.0))
    pos_t_6 = 1.0 / (1.0 + np.exp(-(t_sin - 0.50) / 0.20))
    raw_6 = av_lock * narrow_qrs * rate_6 * age_6 * pos_t_6

    df["ind_class3_avnrt"] = np.sqrt(np.clip(raw_3, 0.0, 1.0))
    df["ind_class4_avrt"] = np.sqrt(np.clip(raw_4, 0.0, 1.0))
    df["ind_class6_saawr"] = np.sqrt(np.clip(raw_6, 0.0, 1.0))
    return df



In [60]:
# 1. Non-Redundant Feature Engineering (Targeted for 3, 4, 6)
# =========================================================
def build_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    for col in [c for c in df.columns if c not in [ID_COL, TARGET_COL]]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df[col].median())

    vr = df["VentricularRate"].clip(20, 300)
    ar = df["AtrialRate"].clip(20, 450)

    # Targeted signals separating AVNRT (3), AVRT (4), and SAAWR (6)
    df["av_log_ratio"] = np.log(ar / vr)                          # 1:1 conduction vs dissociation
    df["st_t_span"] = (df["TOffset"] - df["QOffset"]).clip(1)     # Retrograde P / ST-T window (3 vs 4)

    # Delta-wave widening proxy (QRS duration relative to total ventricular depolarization/repolarization)
    qrs_span = (df["QOffset"] - df["QOnset"]).clip(lower=1)
    qt_span = (df["TOffset"] - df["QOnset"]).clip(lower=1)

    # Clean millisecond QRS-to-QT proportion & bounded AV log-ratio
    df["qrs_qt_ratio"] = df["QRSDuration"].clip(40, 250) / df["QTInterval"].clip(200, 700)
    df["av_log_ratio"] = np.clip(np.log(ar / vr), -0.8, 1.8)

    df["age_rate_prod"] = (df["PatientAge"] / 100.0) * (vr / 100.0)  # Young tachycardia (3,4) vs older (6)
    

    # Circular axis encoding (smooth repolarization & depolarization vectors)
    t_rad, r_rad = np.deg2rad(df["TAxis"]), np.deg2rad(df["RAxis"])
    df["t_sin"], df["t_cos"] = np.sin(t_rad), np.cos(t_rad)
    df["r_sin"], df["r_cos"] = np.sin(r_rad), np.cos(r_rad)
    df["qrs_t_align"] = np.cos(r_rad - t_rad)

    df = add_rare_indicator_features(df)

    # Drop algebraically derived/redundant columns
    drop_cols = ["QRSCount", "QTCorrected", "QOnset", "QOffset", "TOffset", "TAxis", "RAxis"]
    return df.drop(columns=drop_cols, errors="ignore")


# **Model Definition**

In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DeepFeedbackLoopResNet(nn.Module):
    """Configurable deep residual loop block with LayerNorm/BatchNorm and indicator-safe input noise."""

    def __init__(
        self,
        in_dim: int,
        num_classes: int,
        hidden_dim: int = 32,
        num_loop_layers: int = 4,
        max_loops: int = 6,
        exit_conf: float = 0.72,
        dropout: float = 0.25,
        input_noise_std: float = 0.07,
        residual_scale: float = 0.30,
        use_batch_norm: bool = False,
        num_protected_cols: int = 3,
        protected_noise_scale: float = 0.15
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.max_loops = max_loops
        self.exit_conf = exit_conf
        self.input_noise_std = input_noise_std
        self.residual_scale = residual_scale
        self.num_protected_cols = num_protected_cols
        self.protected_noise_scale = protected_noise_scale

        def make_norm(dim: int) -> nn.Module:
            return nn.BatchNorm1d(dim) if use_batch_norm else nn.LayerNorm(dim)

        # Input projection with light feature dropout
        self.in_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            make_norm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout * 0.5) if dropout > 0.0 else nn.Identity(),
        )

        # N-layer residual loop block receiving [x0 + h + prev_probs]
        loop_in_dim = hidden_dim * 2 + num_classes
        layers = []
        for layer_idx in range(num_loop_layers):
            in_ch = loop_in_dim if layer_idx == 0 else hidden_dim
            layers.extend([
                nn.Linear(in_ch, hidden_dim),
                make_norm(hidden_dim),
            ])
            if layer_idx < num_loop_layers - 1:
                layers.append(nn.SiLU())
                if dropout > 0.0:
                    layers.append(nn.Dropout(dropout))

        self.loop_cell = nn.Sequential(*layers)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor):
        if self.training and self.input_noise_std > 0.0:
            noise = torch.randn_like(x) * self.input_noise_std
            # Shield trailing rare-class indicator columns from heavy Gaussian corruption
            if self.num_protected_cols > 0 and x.size(1) > self.num_protected_cols:
                noise[:, -self.num_protected_cols:] *= self.protected_noise_scale
            x = x + noise

        x0 = self.in_proj(x)
        h = x0.clone()
        prev_probs = torch.full(
            (x.size(0), self.num_classes), 1.0 / self.num_classes, device=x.device
        )

        out = torch.zeros(x.size(0), self.num_classes, device=x.device)
        active = torch.ones(x.size(0), dtype=torch.bool, device=x.device)
        step_logits = []

        for step in range(1, self.max_loops + 1):
            loop_input = torch.cat([x0, h, prev_probs], dim=-1)
            h = F.silu(h + self.residual_scale * self.loop_cell(loop_input))
            logits = self.head(h)
            step_logits.append(logits)

            prev_probs = F.softmax(logits, dim=-1)
            done = active & (
                (prev_probs.max(dim=-1).values >= self.exit_conf) | (step == self.max_loops)
            )
            out[done] = logits[done]
            active = active & (~done)

            if not self.training and not active.any():
                break

        return out, step_logits

In [62]:
def confusion_recall_loss(step_logits, targets, cost_matrix, num_classes: int):
    y_oh = F.one_hot(targets, num_classes=num_classes).float()
    loss = 0.0
    for logits in step_logits:
        probs = F.softmax(logits, dim=-1)
        conf_cost = (probs * cost_matrix[targets]).sum(dim=-1).mean()
        tp = (probs * y_oh).sum(dim=0)
        fn = ((1.0 - probs) * y_oh).sum(dim=0)
        fp = (probs * (1.0 - y_oh)).sum(dim=0)
        soft_f15 = (3.25 * tp + 1e-6) / (3.25 * tp + 2.25 * fn + fp + 1e-6)
        loss += 0.5 * conf_cost + 0.5 * (1.0 - soft_f15.mean())
    return loss / len(step_logits)

In [63]:
#depreciated synthetic data imputation augment_rare_anchors

def augment_rare_anchors(
    features: np.ndarray,
    labels: np.ndarray,
    min_samples: int = 60,
    beta_alpha: float = 0.3,
    jitter_scale: float = 0.12,
    seed: int = 42,
):
    """Upsamples rare classes up to min_samples via convex Mixup interpolation and within-class jitter."""
    rng = np.random.default_rng(seed)
    X_list, y_list = [features], [labels]

    for cls in np.unique(labels):
        cls_rows = features[labels == cls]
        n_existing = len(cls_rows)
        if 0 < n_existing < min_samples:
            n_needed = min_samples - n_existing
            # Compute within-class feature spread (safe fallback if only 1 sample exists)
            within_std = cls_rows.std(axis=0, keepdims=True) if n_existing > 1 else np.full((1, features.shape[1]), 0.05)
            within_std = np.where(within_std < 1e-6, 0.05, within_std)

            idx_a = rng.integers(0, n_existing, size=n_needed)
            idx_b = rng.integers(0, n_existing, size=n_needed)
            mix_w = rng.beta(beta_alpha, beta_alpha, size=(n_needed, 1))
            jitter = rng.normal(0.0, jitter_scale, size=(n_needed, features.shape[1])) * within_std

            synth_rows = mix_w * cls_rows[idx_a] + (1.0 - mix_w) * cls_rows[idx_b] + jitter
            X_list.append(synth_rows)
            y_list.append(np.full(n_needed, cls, dtype=labels.dtype))

    return np.vstack(X_list), np.concatenate(y_list)


# **Training Functionality**

In [64]:
#prepare fold data

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler


def prepare_fold_data(
    X_df: pd.DataFrame,
    X_test_df: pd.DataFrame,
    y: np.ndarray,
    tr_idx: np.ndarray,
    va_idx: np.ndarray,
    num_classes: int,
    sampler_power: float = 0.45,
    rarity_power: float = 0.50,
    rarity_max_clip: float = 10.0,
    batch_size: int = 256,
    use_batch_norm: bool = True,
):
    """Scales continuous features on train split while preserving [0, 1] indicators, builds cost matrix, and creates a balanced DataLoader."""
    # 1. Identify binary & [0, 1] indicator columns to pass through unscaled
    passthrough_cols = [
        c for c in X_df.columns
        if c.startswith("ind_") or X_df[c].dropna().isin([0, 1, 0.0, 1.0]).all()
    ]
    scale_cols = [c for c in X_df.columns if c not in passthrough_cols]

    # Fit scaler strictly on continuous training columns while preserving exact column order
    preprocessor = ColumnTransformer(
        transformers=[
            ("scale", RobustScaler(), scale_cols),
            ("pass", "passthrough", passthrough_cols),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    X_tr_scaled = preprocessor.fit_transform(X_df.iloc[tr_idx])[X_df.columns].to_numpy(dtype=np.float32)
    X_va_scaled = preprocessor.transform(X_df.iloc[va_idx])[X_df.columns].to_numpy(dtype=np.float32)
    X_te_scaled = preprocessor.transform(X_test_df)[X_df.columns].to_numpy(dtype=np.float32)

    X_va = torch.tensor(X_va_scaled, dtype=torch.float32, device=device)
    X_te = torch.tensor(X_te_scaled, dtype=torch.float32, device=device)
    y_tr_raw, y_va = y[tr_idx], y[va_idx]

    # 2. Compute class frequencies and asymmetric confusion cost matrix
    counts = np.bincount(y_tr_raw, minlength=num_classes)
    rarity_pen = np.clip((counts.max() / np.maximum(counts, 1)) ** rarity_power, 1.0, rarity_max_clip)
    cost_mat = np.ones((num_classes, num_classes), dtype=np.float32)
    np.fill_diagonal(cost_mat, 0.0)
    cost_tensor = torch.tensor(cost_mat * rarity_pen[:, None], device=device)

    # 3. Build balanced WeightedRandomSampler DataLoader so minority classes appear regularly
    sample_weights = (1.0 / np.maximum(counts[y_tr_raw], 1)) ** sampler_power
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_tr_raw), replacement=True)
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32), torch.tensor(y_tr_raw, dtype=torch.long)),
        batch_size=batch_size,
        sampler=sampler,
        drop_last=use_batch_norm,  # Prevents 1-sample batch crashes when BatchNorm1d is active
    )

    in_dim = X_tr_scaled.shape[1]
    return loader, X_va, y_va, X_te, cost_tensor, in_dim

#Selective Scaling via ColumnTransformer: Instead of running RobustScaler across every column, continuous columns get robustly scaled (Q25–Q75) while binary columns (e.g., Gender) and rare indicator columns (ind_class3_avnrt, ind_class4_avrt, ind_class6_saawr) pass straight through unchanged.
#Zero-IQR Protection: If any continuous column happens to have an IQR of 0 on a particular fold split, RobustScaler won't distort its scale relative to the rest of the features, and column order is strictly preserved so your feat_names in the live weight graph stay 100% aligned.


In [65]:
#fit_single_fold

#integrates L1 weight sparsity, CosineAnnealingWarmRestarts (sched.step()), 
#the Min-Class Floor + Macro F1 selection criterion with the val_loss <= best_val_loss * val_loss_guard overfitting guard, 
#and in-place live dashboard updates.

import copy
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score
from sklearn.preprocessing import label_binarize


def _run_train_epoch(
    model: torch.nn.Module,
    loader,
    opt: torch.optim.Optimizer,
    cost_tensor: torch.Tensor,
    num_classes: int,
    l1_lambda: float,
) -> float:
    """Runs one training epoch with optional L1 weight sparsity and returns mean batch loss."""
    model.train()
    running_loss = 0.0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        opt.zero_grad()
        _, step_logits = model(bx)
        loss = confusion_recall_loss(step_logits, by, cost_tensor, num_classes)
        if l1_lambda > 0.0:
            loss = loss + l1_lambda * sum(
                p.abs().sum() for p in model.parameters() if p.requires_grad and p.ndim > 1
            )
        loss.backward()
        opt.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)


def _evaluate_validation(
    model: torch.nn.Module,
    X_va: torch.Tensor,
    y_va: np.ndarray,
    y_va_t: torch.Tensor,
    y_va_bin: np.ndarray,
    cost_tensor: torch.Tensor,
    num_classes: int,
    rare_cls_idx: np.ndarray,
    macro_weight: float,
    rare_f1_weight: float,
    pr_auc_weight: float,
) -> dict:
    """Evaluates validation metrics and returns a snapshot dictionary."""
    model.eval()
    with torch.no_grad():
        va_logits, va_step_logits = model(X_va)
        val_loss = float(confusion_recall_loss(va_step_logits, y_va_t, cost_tensor, num_classes).item())
        va_p = F.softmax(va_logits, dim=-1).cpu().numpy()
        va_preds = va_p.argmax(axis=1)

        cls_f1 = f1_score(y_va, va_preds, average=None, labels=range(num_classes), zero_division=0)
        val_f1 = float(cls_f1.mean())
        bot4_f1 = float(cls_f1[rare_cls_idx].mean())
        macro_pr_auc = float(average_precision_score(y_va_bin, va_p, average="macro"))
        sel_score = (macro_weight * val_f1) + (rare_f1_weight * bot4_f1) + (pr_auc_weight * macro_pr_auc)

    return {
        "val_loss": val_loss,
        "va_p": va_p,
        "va_preds": va_preds,
        "cls_f1": cls_f1,
        "f1": val_f1,
        "bot4_f1": bot4_f1,
        "pr_auc": macro_pr_auc,
        "sel": sel_score,
    }


def _should_stop_training(
    ep: int,
    min_epochs: int,
    stale: int,
    patience: int,
    loss_ratio: float,
    max_overfit_ratio: float = 1.45,
) -> tuple[bool, str]:
    """Checks early stopping via metric plateau or sustained train/val loss divergence."""
    if ep < min_epochs:
        return False, ""
    if stale >= patience:
        return True, f"Metric Plateau — no selection improvement for {stale} epochs (patience={patience})"
    if loss_ratio > max_overfit_ratio and stale >= max(patience // 2, 20):
        return True, f"Overfit Guard — Va/Tr ratio hit {loss_ratio:.2f}x (> {max_overfit_ratio:.2f}x) with {stale} stale epochs"
    return False, ""


def _compact_tag(text: str) -> str:
    """Applies compact short_title formatting."""
    return (
        text.replace("Fold ", "F")
        .replace(" Ep ", "E")
        .replace("MacroF1: ", "mF1:")
        .replace("Bot4F1: ", "b4:")
        .replace("PR-AUC: ", "PR:")
        .replace("Tr/Va: ", "L:")
        .replace(" | ", " ")
        .replace("0.", ".")
        .split(" (")[0]
    )


def print_stop_reason(
    fold: int,
    ep: int,
    max_epochs: int,
    reason: str,
    best_ep: int,
    best_sel: float,
    best_stats: dict,
) -> str:
    """Prints a compact 2-line fold stop summary and returns a single-line status for Groq."""
    short_reason = (
        reason.split("—")[0].strip()
        if reason
        else f"MaxEp ({ep}/{max_epochs})"
    )
    line1 = f"F{fold + 1} Stop E{ep:03d}: {short_reason}"
    line2 = _compact_tag(
        f"Fold {fold + 1} Ep {best_ep:03d} | "
        f"MacroF1: {best_stats['f1']:.3f} | "
        f"Bot4F1: {best_stats['bot4_f1']:.3f} | "
        f"PR-AUC: {best_stats['pr_auc']:.3f}"
    )
    print(f"{line1}\n{line2}")
    return f"{line2} ({short_reason})"



def fit_single_fold(
    fold: int,
    model: torch.nn.Module,
    opt: torch.optim.Optimizer,
    sched: torch.optim.lr_scheduler._LRScheduler,
    loader,
    X_va: torch.Tensor,
    y_va: np.ndarray,
    X_te: torch.Tensor,
    cost_tensor: torch.Tensor,
    class_names: np.ndarray,
    feat_names: list,
    l1_lambda: float = 1e-4,
    max_epochs: int = 240,
    patience: int = 80,
    min_epochs: int = 100,
    log_interval: int = 20,
    top_percentile: int = 80,
    macro_weight: float = 0.50,
    rare_f1_weight: float = 0.25,
    pr_auc_weight: float = 0.25,
    val_loss_guard: float = 1.15,
    max_overfit_ratio: float = 1.45,
    show_dashboard_in:bool = False
):
    """Trains a single fold using modular train, validation, checkpointing, and stopping helpers."""
    num_classes = len(class_names)
    y_va_t = torch.tensor(y_va, dtype=torch.long, device=device)
    y_va_bin = label_binarize(y_va, classes=range(num_classes))

    va_counts = np.bincount(y_va, minlength=num_classes)
    rare_cls_idx = np.argsort(va_counts)[:4]
    rare_mask = np.isin(y_va, rare_cls_idx)

    best_sel, best_val_loss, best_ep, stale = -1.0, float("inf"), 1, 0
    best_stats, stop_reason, dash_handle = {}, "", None
    history = {k: [] for k in ["ep", "tr_loss", "va_loss", "loss_ratio", "va_macro_f1", "rare_true_p", "overconf_gap", "w_norm"]}

    for ep in range(1, max_epochs + 1):
        # 1. Train epoch & step cosine schedule
        tr_loss_ep = _run_train_epoch(model, loader, opt, cost_tensor, num_classes, l1_lambda)
        sched.step()

        # 2. Evaluate validation & update peak checkpoint
        ev = _evaluate_validation(
            model, X_va, y_va, y_va_t, y_va_bin, cost_tensor, num_classes,
            rare_cls_idx, macro_weight, rare_f1_weight, pr_auc_weight
        )
        loss_ratio = ev["val_loss"] / max(tr_loss_ep, 1e-8)
        best_val_loss = min(best_val_loss, ev["val_loss"])

        if ev["sel"] > best_sel and ev["val_loss"] <= best_val_loss * val_loss_guard:
            best_sel, best_ep, stale = ev["sel"], ep, 0
            with torch.no_grad():
                te_logits, _ = model(X_te)
            best_stats = {
                "f1": ev["f1"],
                "bot4_f1": ev["bot4_f1"],
                "pr_auc": ev["pr_auc"],
                "cls_f1": ev["cls_f1"].copy(),
                "va_p": ev["va_p"],
                "te_p": F.softmax(te_logits, dim=-1).cpu().numpy(),
                "weights": copy.deepcopy(model.state_dict()),
            }
        else:
            stale += 1

        # 3. Record history & refresh 3x2 live dashboard
        if ep % log_interval == 0 or ep == 1:
            overconf_gap = float(ev["va_p"].max(axis=1).mean() - (ev["va_preds"] == y_va).mean())
            rare_true_p = float(ev["va_p"][rare_mask, y_va[rare_mask]].mean()) if rare_mask.any() else 0.0
            w_norm = sum(p.norm(2).item() for p in model.parameters() if p.requires_grad)

            for k, v in zip(history.keys(), [ep, tr_loss_ep, ev["val_loss"], loss_ratio, ev["f1"], rare_true_p, overconf_gap, w_norm]):
                history[k].append(v)

            dash_title = (
                f"Fold {fold + 1} Ep {ep:03d} | MacroF1: {ev['f1']:.3f} | Bot4F1: {ev['bot4_f1']:.3f} | "
                f"PR-AUC: {ev['pr_auc']:.3f} | Tr/Va: {tr_loss_ep:.3f}/{ev['val_loss']:.3f} ({loss_ratio:.2f}x)"
            )
            
            dash_handle = plot_live_fold_dashboard(
                cls_f1=ev["cls_f1"],
                class_names=class_names,
                model=model,
                feat_names=feat_names,
                title=dash_title,
                top_percentile=top_percentile,
                history=history,
                handle=dash_handle,
                y_true=y_va,
                va_probs=ev["va_p"],
                show_dashboard=show_dashboard_in,
            )


        # 4. Check early stopping conditions
        stop_flag, stop_reason = _should_stop_training(ep, min_epochs, stale, patience, loss_ratio, max_overfit_ratio)
        if stop_flag:
            break

    # 5. Restore best checkpoint & print stop summary
    if not best_stats:
        with torch.no_grad():
            te_p = F.softmax(model(X_te)[0], dim=-1).cpu().numpy()
        best_stats = {
            "f1": ev["f1"], "bot4_f1": ev["bot4_f1"], "pr_auc": ev["pr_auc"],
            "cls_f1": ev["cls_f1"], "va_p": ev["va_p"], "te_p": te_p, "weights": None
        }

    if best_stats["weights"] is not None:
        model.load_state_dict(best_stats["weights"])

    print_stop_reason(ep, max_epochs, stop_reason, best_ep, best_sel, best_stats)

    final_title = (
        f"Fold {fold + 1} COMPLETE (Best Ep {best_ep:03d}) | MacroF1: {best_stats['f1']:.3f} | "
        f"Bot4F1: {best_stats['bot4_f1']:.3f} | PR-AUC: {best_stats['pr_auc']:.3f}"
    )

    dash_handle = plot_live_fold_dashboard(
        cls_f1=best_stats["cls_f1"],
        class_names=class_names,
        model=model,
        feat_names=feat_names,
        title=final_title,
        top_percentile=top_percentile,
        history=history,
        handle=dash_handle,
        y_true=y_va,
        va_probs=best_stats["va_p"],
        show_dashboard=show_dashboard_in,
    )

    return best_stats["va_p"], best_stats["te_p"], model



In [66]:
#build_loop_model_and_opt        AdamW L2 optimizer and CosineAnnealingWarmRestarts


def build_loop_model_and_opt(
    in_dim: int,
    num_classes: int,
    hidden_dim: int = 32,
    num_loop_layers: int = 4,
    max_loops: int = 6,
    exit_conf: float = 0.72,
    dropout: float = 0.25,
    input_noise_std: float = 0.07,
    residual_scale: float = 0.30,
    use_batch_norm: bool = False,
    lr: float = 2.5e-3,
    weight_decay: float = 5e-2,
    t_0: int = 40,
    t_mult: int = 2,
    eta_min: float = 1e-5,
    num_protected_cols: int = 3,
    protected_noise_scale: float = 0.15,
):
    """Instantiates DeepFeedbackLoopResNet, AdamW (L2 decay), and CosineAnnealingWarmRestarts scheduler."""
    model = DeepFeedbackLoopResNet(
        in_dim=in_dim,
        num_classes=num_classes,
        hidden_dim=hidden_dim,
        num_loop_layers=num_loop_layers,
        max_loops=max_loops,
        exit_conf=exit_conf,
        dropout=dropout,
        input_noise_std=input_noise_std,
        residual_scale=residual_scale,
        use_batch_norm=use_batch_norm,
        num_protected_cols=num_protected_cols,
        protected_noise_scale=protected_noise_scale,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=t_0, T_mult=t_mult, eta_min=eta_min
    )

    return model, opt, sched


In [67]:
#tune_oof_multipliers

#performs coordinate descent over out-of-fold probability multipliers from the rarest class to the most common class while capping predicted counts on ultra-rare classes to prevent false-positive spikes.

import numpy as np
from sklearn.metrics import f1_score


def tune_oof_multipliers(
    oof_probs: np.ndarray,
    y: np.ndarray,
    num_classes: int,
    tune_passes: int = 4,
    mult_min: float = 0.2,
    mult_max: float = 6.0,
    mult_steps: int = 60,
    rare_support_thresh: int = 15,
    rare_max_mult: int = 2,
    rare_min_preds: int = 4,
) -> np.ndarray:
    """Calibrates per-class probability multipliers on OOF predictions via coordinate descent."""
    mult = np.ones(num_classes, dtype=np.float64)
    all_counts = np.bincount(y, minlength=num_classes)
    candidates = np.geomspace(mult_min, mult_max, mult_steps)

    # Optimize classes in ascending order of true support (rarest -> most common)
    class_order = np.argsort(all_counts)

    for _ in range(tune_passes):
        for c in class_order:
            best_m = mult[c]
            best_s = f1_score(
                y, np.argmax(oof_probs * mult, axis=1), average="macro", zero_division=0
            )
            max_allowed_preds = max(all_counts[c] * rare_max_mult, rare_min_preds)

            for cand in candidates:
                m_try = mult.copy()
                m_try[c] = cand
                preds = np.argmax(oof_probs * m_try, axis=1)

                # Guardrail: prevent ultra-rare classes from over-predicting false positives
                if all_counts[c] <= rare_support_thresh and np.sum(preds == c) > max_allowed_preds:
                    continue

                score = f1_score(y, preds, average="macro", zero_division=0)
                if score > best_s:
                    best_s = score
                    best_m = cand

            mult[c] = best_m

    return mult


# *Model HyperParameters (Training Main Loop)*

In [68]:
#def train_and_save_ensemble

import json
import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, RobustScaler
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def prepare_fold_train_val(
    X_df: pd.DataFrame,
    y: np.ndarray,
    tr_idx: np.ndarray,
    va_idx: np.ndarray,
    num_classes: int,
    sampler_power: float = 0.62,
    rarity_power: float = 0.62,
    rarity_max_clip: float = 12.0,
    batch_size: int = 256,
    use_batch_norm: bool = False,
):
    """Fits selective RobustScaler on train split, builds cost matrix, and returns DataLoader + fitted preprocessor."""
    passthrough_cols = [
        c for c in X_df.columns
        if c.startswith("ind_") or X_df[c].dropna().isin([0, 1, 0.0, 1.0]).all()
    ]
    scale_cols = [c for c in X_df.columns if c not in passthrough_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("scale", RobustScaler(), scale_cols),
            ("pass", "passthrough", passthrough_cols),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    X_tr_scaled = preprocessor.fit_transform(X_df.iloc[tr_idx])[X_df.columns].to_numpy(dtype=np.float32)
    X_va_scaled = preprocessor.transform(X_df.iloc[va_idx])[X_df.columns].to_numpy(dtype=np.float32)
    X_va = torch.tensor(X_va_scaled, dtype=torch.float32, device=device)
    y_tr_raw, y_va = y[tr_idx], y[va_idx]

    counts = np.bincount(y_tr_raw, minlength=num_classes)
    rarity_pen = np.clip((counts.max() / np.maximum(counts, 1)) ** rarity_power, 1.0, rarity_max_clip)
    cost_mat = np.ones((num_classes, num_classes), dtype=np.float32)
    np.fill_diagonal(cost_mat, 0.0)
    cost_tensor = torch.tensor(cost_mat * rarity_pen[:, None], device=device)

    sample_weights = (1.0 / np.maximum(counts[y_tr_raw], 1)) ** sampler_power
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_tr_raw), replacement=True)
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32), torch.tensor(y_tr_raw, dtype=torch.long)),
        batch_size=batch_size,
        sampler=sampler,
        drop_last=use_batch_norm,
    )

    return loader, X_va, y_va, cost_tensor, preprocessor


def train_and_save_ensemble(
    X_df: pd.DataFrame,
    y_raw: np.ndarray,
    params: dict,
    artifact_path: str = "loop_resnet_ensemble.pt",
    config_json_path: str = "train_config_report.json",
    show_dashboard: bool = False,
):
    """Trains 5-fold LoopResNet from a params dictionary and serializes weights, scalers, multipliers & settings."""
    t_start = time.time()
    feat_names = list(X_df.columns)
    in_dim = len(feat_names)

    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)
    num_classes = len(encoder.classes_)

    # Merge user overrides with clean defaults
    cfg = {
        "hidden_dim": 32,
        "num_loop_layers": 4,
        "max_loops": 6,
        "exit_conf": 0.72,
        "dropout": 0.25,
        "input_noise_std": 0.07,
        "protected_noise_scale": 0.15,
        "num_protected_cols": 3,
        "residual_scale": 0.30,
        "use_batch_norm": False,
        "n_splits": 5,
        "random_state": 42,
        "sampler_power": 0.62,
        "rarity_power": 0.62,
        "rarity_max_clip": 12.0,
        "l1_lambda": 0.0,
        "weight_decay": 5e-2,
        "batch_size": 256,
        "lr": 2.5e-3,
        "t_0": 40,
        "t_mult": 2,
        "eta_min": 1e-5,
        "max_epochs": 280,
        "patience": 100,
        "min_epochs": 120,
        "max_overfit_ratio": 1.45,
        "log_interval": 20,
        "top_percentile": 80,
        "macro_weight": 0.35,
        "rare_f1_weight": 0.40,
        "pr_auc_weight": 0.25,
        "val_loss_guard": 1.20,
        "tune_passes": 5,
        "mult_min": 0.2,
        "mult_max": 30.0,
        "mult_steps": 80,
        "rare_support_thresh": 90,
        "rare_max_mult": 3.0,
        "rare_min_preds": 2,
    }
    cfg.update(params)

    skf = StratifiedKFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["random_state"])
    oof_probs = np.zeros((len(X_df), num_classes), dtype=np.float64)
    fold_artifacts = []

    if show_dashboard:
        print_dashboard_legend()

    # Dummy 1-row tensor just to satisfy fit_single_fold's X_te signature without touching Test.csv
    dummy_te = torch.zeros((1, in_dim), dtype=torch.float32, device=device)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y)):
        print(f"\n--- Fold {fold + 1}/{cfg['n_splits']} ---")

        loader, X_va, y_va, cost_tensor, preprocessor = prepare_fold_train_val(
            X_df, y, tr_idx, va_idx, num_classes,
            sampler_power=cfg["sampler_power"],
            rarity_power=cfg["rarity_power"],
            rarity_max_clip=cfg["rarity_max_clip"],
            batch_size=cfg["batch_size"],
            use_batch_norm=cfg["use_batch_norm"],
        )

        model, opt, sched = build_loop_model_and_opt(
            in_dim=in_dim,
            num_classes=num_classes,
            hidden_dim=cfg["hidden_dim"],
            num_loop_layers=cfg["num_loop_layers"],
            max_loops=cfg["max_loops"],
            exit_conf=cfg["exit_conf"],
            dropout=cfg["dropout"],
            input_noise_std=cfg["input_noise_std"],
            residual_scale=cfg["residual_scale"],
            use_batch_norm=cfg["use_batch_norm"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            t_0=cfg["t_0"],
            t_mult=cfg["t_mult"],
            eta_min=cfg["eta_min"],
            num_protected_cols=cfg["num_protected_cols"],
            protected_noise_scale=cfg["protected_noise_scale"],
        )

        best_va_p, _, model = fit_single_fold(
            fold, model, opt, sched, loader, X_va, y_va, dummy_te, cost_tensor,
            encoder.classes_, feat_names,
            l1_lambda=cfg["l1_lambda"],
            max_epochs=cfg["max_epochs"],
            patience=cfg["patience"],
            min_epochs=cfg["min_epochs"],
            log_interval=cfg["log_interval"],
            top_percentile=cfg["top_percentile"],
            macro_weight=cfg["macro_weight"],
            rare_f1_weight=cfg["rare_f1_weight"],
            pr_auc_weight=cfg["pr_auc_weight"],
            val_loss_guard=cfg["val_loss_guard"],
            max_overfit_ratio=cfg["max_overfit_ratio"],
        )

        oof_probs[va_idx] = best_va_p
        fold_artifacts.append({
            "fold": fold,
            "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "preprocessor": preprocessor,
        })

    # Tune OOF probability multipliers
    mult = tune_oof_multipliers(
        oof_probs, y, num_classes,
        tune_passes=cfg["tune_passes"],
        mult_min=cfg["mult_min"],
        mult_max=cfg["mult_max"],
        mult_steps=cfg["mult_steps"],
        rare_support_thresh=cfg["rare_support_thresh"],
        rare_max_mult=cfg["rare_max_mult"],
        rare_min_preds=cfg["rare_min_preds"],
    )

    oof_preds = encoder.inverse_transform(np.argmax(oof_probs * mult, axis=1))
    macro_f1 = float(f1_score(y_raw, oof_preds, average="macro"))
    elapsed = round(time.time() - t_start, 2)

    # Save complete inference bundle (.pt) + readable settings & metrics report (.json)
    bundle = {
        "params": cfg,
        "in_dim": in_dim,
        "feat_names": feat_names,
        "classes": encoder.classes_.tolist(),
        "multipliers": mult.tolist(),
        "oof_macro_f1": macro_f1,
        "folds": fold_artifacts,
    }
    torch.save(bundle, artifact_path)

    report = {
        "artifact_path": artifact_path,
        "oof_macro_f1": round(macro_f1, 5),
        "elapsed_sec": elapsed,
        "classes": encoder.classes_.tolist(),
        "multipliers": {str(c): round(float(m), 4) for c, m in zip(encoder.classes_, mult)},
        "params": cfg,
    }
    with open(config_json_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\nSaved ensemble weights & scalers -> {artifact_path}")
    print(f"Saved training settings & metrics -> {config_json_path}")
    print(f"OOF Mean F1 (Macro): {macro_f1:.4f}")
    print(classification_report(y_raw, oof_preds, digits=4))

    return oof_preds, oof_probs, bundle


# **Prediction**

In [69]:
def predict_from_ensemble(
    X_test_df: pd.DataFrame,
    test_ids: pd.Series,
    artifact_path: str = "loop_resnet_ensemble.pt",
    submission_path: str = "submission.csv",
    id_col: str = "patient_ID",
    target_col: str = "Rhythm",
) -> pd.DataFrame:
    """Loads serialized 5-fold LoopResNet ensemble + scalers and generates Test.csv predictions."""
    bundle = torch.load(artifact_path, map_location=device, weights_only=False)
    cfg = bundle["params"]
    feat_names = bundle["feat_names"]
    classes = np.array(bundle["classes"])
    mult = np.array(bundle["multipliers"], dtype=np.float64)
    num_classes = len(classes)

    # Ensure exact feature column alignment
    X_test_aligned = X_test_df[feat_names]
    test_probs = np.zeros((len(X_test_aligned), num_classes), dtype=np.float64)

    for fold_data in bundle["folds"]:
        # 1. Transform test features using this fold's fitted scaler
        X_te_scaled = fold_data["preprocessor"].transform(X_test_aligned)[feat_names].to_numpy(dtype=np.float32)
        X_te_t = torch.tensor(X_te_scaled, dtype=torch.float32, device=device)

        # 2. Rebuild model architecture from saved config and load fold weights
        model = DeepFeedbackLoopResNet(
            in_dim=bundle["in_dim"],
            num_classes=num_classes,
            hidden_dim=cfg["hidden_dim"],
            num_loop_layers=cfg["num_loop_layers"],
            max_loops=cfg["max_loops"],
            exit_conf=cfg["exit_conf"],
            dropout=cfg["dropout"],
            input_noise_std=cfg["input_noise_std"],
            residual_scale=cfg["residual_scale"],
            use_batch_norm=cfg["use_batch_norm"],
            num_protected_cols=cfg["num_protected_cols"],
            protected_noise_scale=cfg["protected_noise_scale"],
        ).to(device)
        model.load_state_dict(fold_data["state_dict"])
        model.eval()

        # 3. Accumulate fold probabilities
        with torch.no_grad():
            logits, _ = model(X_te_t)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            test_probs += probs / float(len(bundle["folds"]))

    # 4. Apply saved OOF class multipliers and save submission
    test_preds = classes[np.argmax(test_probs * mult, axis=1)]
    sub = pd.DataFrame({id_col: test_ids, target_col: test_preds})
    sub.to_csv(submission_path, index=False)

    print(f"Loaded {artifact_path} (OOF Macro F1: {bundle['oof_macro_f1']:.4f})")
    print(f"Saved {submission_path}! Predicted Test Counts:\n{sub[target_col].value_counts()}")
    return sub


# **LLM Controlled Parameter Search**

In [70]:
def ask_groq_for_next_params(
    trial_history: list,
    fallback_params: dict,
    reasoning_log_path: str = "groq_reasoning_log.txt",
) -> tuple[dict, str]:
    """Queries Groq for next hyperparameters, prints reasoning/diff, and appends to a log file."""
    schema_desc = {k: f"{dtype.__name__} in [{lo}, {hi}]" for k, (lo, hi, dtype) in ALLOWED_BOUNDS.items()}
    prompt = (
        "You are an expert ML engineer optimizing a 5-fold DeepFeedbackLoopResNet for an 11-class "
        "imbalanced ECG arrhythmia dataset (Classes 3, 4, 6 have 5-11 rows; Class 2 has 85 rows; "
        "majority classes 1, 7, 8, 9 have 1100-2700 rows).\n\n"
        "Analyze the chronological trial history below (per-class F1, PR-AUC, multipliers, and params). "
        "Diagnose which classes are bottlenecking oof_macro_f1 (low PR-AUC = model separation issue; "
        "high PR-AUC + low F1 = multiplier/guardrail issue) and propose a distinct, improved config.\n\n"
        f"Allowed parameter schema:\n{json.dumps(schema_desc, indent=2)}\n\n"
        f"Trial History:\n{json.dumps(trial_history, indent=2)}\n\n"
        "Respond with ONLY a valid JSON object containing three keys:\n"
        '1. "hypothesis": 1-sentence summary of the primary goal for this trial\n'
        '2. "reasoning": list of 3-5 bullet strings explaining which classes struggled and why each changed parameter helps\n'
        '3. "params": dictionary containing all 13 hyperparameter keys above'
    )

    next_trial_num = len(trial_history) + 1

    for model_name in ["openai/gpt-oss-120b", "qwen/qwen3.8-27b", "openai/gpt-oss-20b"]:
        try:
            resp = groq_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "You are an ML hyperparameter optimizer. Output valid JSON only."},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.35,
            )
            # Fix: index [0] into resp.choices list
            raw_content = resp.choices[0].message.content
            data = json.loads(raw_content)

            next_params = sanitize_params(data.get("params", {}), fallback_params)
            hypothesis = str(data.get("hypothesis", "No hypothesis provided."))
            raw_reasoning = data.get("reasoning", [])
            if isinstance(raw_reasoning, list):
                reasoning_lines = [f"  - {str(item)}" for item in raw_reasoning]
            else:
                reasoning_lines = [f"  - {str(raw_reasoning)}"]

            # Build formatted report block
            report_lines = [
                "=" * 74,
                f"TRIAL {next_trial_num} PROPOSAL | Model: {model_name} | Prev Best F1: {trial_history[-1]['oof_macro_f1']:.5f}",
                f"Hypothesis : {hypothesis}",
                "Reasoning  :",
                *reasoning_lines,
                "-" * 74,
                f"{'Parameter':<24} | {'Prev Best':>10} | {'Next Trial':>10} | {'Status'}",
                "-" * 74,
            ]
            for k in ALLOWED_BOUNDS:
                old_v, new_v = fallback_params.get(k), next_params.get(k)
                status = "CHANGED *" if old_v != new_v else "unchanged"
                report_lines.append(f"{k:<24} | {str(old_v):>10} | {str(new_v):>10} | {status}")
            report_lines.append("=" * 74 + "\n")

            report_text = "\n".join(report_lines)
            print("\n" + report_text)

            # Append permanently to reasoning log file
            with open(reasoning_log_path, "a") as f:
                f.write(report_text + "\n")

            return next_params, f"{hypothesis} | {' '.join(reasoning_lines)}"
        except Exception as e:
            print(f"[Warning] {model_name} failed ({e}), trying fallback...")

    return fallback_params.copy(), "Fallback to previous best parameters."


In [71]:
#data for training
DATA_DIR = "/kaggle/input/competitions/fall-2026-csulb-pr-in-class-competiton"
ID_COL, TARGET_COL = "patient_ID", "Rhythm"

# =========================================================
# DATA LOADING & STAGE A / STAGE B SUB-FUNCTIONS
# =========================================================
def load_train_data(plot_eda: bool = False):
    """Reads Train.csv, runs feature engineering, and returns (X_df, y_raw)."""
    train_df = pd.read_csv(f"{DATA_DIR}/Train.csv")
    X_df = build_features(train_df.drop(columns=[ID_COL, TARGET_COL]))
    y_raw = train_df[TARGET_COL].astype(str).values

    if plot_eda:
        plot_indicator_grid(X_df, y_raw)
        sep_ranking = plot_rare_distributions(X_df, y_raw, rare_codes=("3", "4", "6"))
        print(sep_ranking)

    return X_df, y_raw

In [72]:
# 1. Initialize OpenAI-compatible client pointing to Groq's API endpoint
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import requests
api_key=UserSecretsClient().get_secret("GroqLLMCallKey")
groq_client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",   # <- correct Groq endpoint
)

# 2. List available Groq models
models_resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {api_key}"},
).json()
print("Available Models:", [m["id"] for m in models_resp.get("data", [])] )


# 3. Test chat completion
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with 'Connected' if you can read this."}],
)
print("Groq Status:", response.choices[0].message.content.strip())

Available Models: ['openai/gpt-oss-20b', 'whisper-large-v3-turbo', 'allam-2-7b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-safeguard-20b', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-arabic-saudi', 'qwen/qwen3.8-27b', 'meta-llama/llama-prompt-guard-2-86m', 'whisper-large-v3']
Groq Status: Connected


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score
from sklearn.preprocessing import label_binarize



ALLOWED_BOUNDS = {
    "hidden_dim": (24, 64, int),
    "dropout": (0.05, 0.40, float),
    "input_noise_std": (0.01, 0.12, float),
    "protected_noise_scale": (0.0, 0.50, float),
    "residual_scale": (0.15, 0.50, float),
    "exit_conf": (0.60, 0.90, float),
    "sampler_power": (0.45, 0.80, float),
    "rarity_power": (0.45, 0.85, float),
    "rarity_max_clip": (8.0, 24.0, float),
    "weight_decay": (1e-3, 1e-1, float),
    "lr": (5e-4, 5e-3, float),
    "rare_support_thresh": (15, 120, int),
    "rare_max_mult": (1.5, 6.0, float),
}




def train(
    train_params: dict,
    X_df: pd.DataFrame = None,
    y_raw: np.ndarray = None,
    artifact_path: str = "loop_resnet_ensemble.pt",
    config_json_path: str = "train_config_report.json",
    show_dashboard: bool = True,
):
    """Runs 5-fold training using train_params (loads Train.csv if X_df/y_raw are not passed)."""
    if X_df is None or y_raw is None:
        X_df, y_raw = load_train_data(plot_eda=show_dashboard)

    return train_and_save_ensemble(
        X_df=X_df,
        y_raw=y_raw,
        params=train_params,
        artifact_path=artifact_path,
        config_json_path=config_json_path,
        show_dashboard=show_dashboard,
    )


def inference(
    artifact_path: str = "loop_resnet_ensemble.pt",
    submission_path: str = "submission.csv",
) -> pd.DataFrame:
    """Loads saved .pt ensemble + scalers, transforms Test.csv, and writes submission.csv."""
    test_df = pd.read_csv(f"{DATA_DIR}/Test.csv")
    X_test_df = build_features(test_df.drop(columns=[ID_COL]))

    return predict_from_ensemble(
        X_test_df=X_test_df,
        test_ids=test_df[ID_COL],
        artifact_path=artifact_path,
        submission_path=submission_path,
        id_col=ID_COL,
        target_col=TARGET_COL,
    )


# =========================================================
# LLM SUPERVISOR & SEARCH SUB-FUNCTIONS
# =========================================================
def sanitize_params(raw_params: dict, fallback: dict) -> dict:
    """Clamps LLM-proposed hyperparameters strictly inside safe numeric bounds."""
    clean = fallback.copy()
    for k, (lo, hi, dtype) in ALLOWED_BOUNDS.items():
        if k in raw_params:
            try:
                clean[k] = dtype(np.clip(float(raw_params[k]), lo, hi))
            except (ValueError, TypeError):
                pass
    return clean


def build_trial_diagnostics(
    trial: int,
    params: dict,
    hypothesis: str,
    oof_preds: np.ndarray,
    oof_probs: np.ndarray,
    y_raw: np.ndarray,
    bundle: dict,
) -> dict:
    """Computes per-class F1, PR-AUC, and multipliers into a compact summary for Groq."""
    classes = np.array(bundle["classes"])
    num_classes = len(classes)
    y_true = np.searchsorted(classes, y_raw)
    y_pred = np.searchsorted(classes, oof_preds)

    cls_f1 = f1_score(y_true, y_pred, average=None, labels=range(num_classes), zero_division=0)
    y_bin = label_binarize(y_true, classes=range(num_classes))
    cls_ap = [average_precision_score(y_bin[:, c], oof_probs[:, c]) for c in range(num_classes)]

    return {
        "trial": trial,
        "oof_macro_f1": round(float(np.mean(cls_f1)), 5),
        "oof_macro_pr_auc": round(float(np.mean(cls_ap)), 5),
        "hypothesis": hypothesis,
        "params": params,
        "per_class_metrics": {
            str(classes[c]): {
                "f1": round(float(cls_f1[c]), 4),
                "pr_auc": round(float(cls_ap[c]), 4),
                "mult": round(float(bundle["multipliers"][c]), 3),
            }
            for c in range(num_classes)
        },
    }


def run_groq_driven_search(
    initial_params: dict,
    n_trials: int = 5,
    best_artifact_path: str = "best_loop_resnet_ensemble.pt",
    history_json_path: str = "groq_search_history.json",
    show_dashboard: bool = False,
):
    """Orchestrates LLM-supervised hyperparameter search by calling load_train_data, train, and ask_groq."""
    X_df, y_raw = load_train_data(plot_eda=False)
    trial_history = []
    best_f1 = -1.0
    best_params = initial_params.copy()
    current_params = initial_params.copy()
    current_hypothesis = "Baseline seed configuration."

    for trial in range(1, n_trials + 1):
        print(f"\n{'#'*72}")
        print(f"STARTING GROQ-DRIVEN TRIAL {trial}/{n_trials}")
        print(f"Reason : {current_hypothesis}")
        print(f"Params : {json.dumps(current_params, indent=2)}")
        print(f"{'#'*72}")

        tmp_pt = f"trial_{trial}_ensemble.pt"
        tmp_json = f"trial_{trial}_report.json"

        oof_preds, oof_probs, bundle = train(
            train_params=current_params,
            X_df=X_df,
            y_raw=y_raw,
            artifact_path=tmp_pt,
            config_json_path=tmp_json,
            show_dashboard=show_dashboard,
        )

        diag = build_trial_diagnostics(
            trial, current_params, current_hypothesis, oof_preds, oof_probs, y_raw, bundle
        )
        trial_history.append(diag)

        with open(history_json_path, "w") as f:
            json.dump(trial_history, f, indent=2)

        if diag["oof_macro_f1"] > best_f1:
            best_f1 = diag["oof_macro_f1"]
            best_params = current_params.copy()
            os.replace(tmp_pt, best_artifact_path)
            print(f">>> NEW BEST MODEL SAVED ({best_artifact_path}) | OOF Macro F1: {best_f1:.5f} <<<")
        else:
            os.remove(tmp_pt)

        if trial < n_trials:
            current_params, current_hypothesis = ask_groq_for_next_params(trial_history, best_params)

    print(f"\nSearch Complete! Best OOF Macro F1: {best_f1:.5f} -> {best_artifact_path}")
    return trial_history


# =========================================================
# RUN SEARCH & GENERATE FINAL SUBMISSION
# =========================================================
seed_params = {
    "hidden_dim": 40,
    "dropout": 0.18,
    "input_noise_std": 0.05,
    "protected_noise_scale": 0.15,
    "residual_scale": 0.30,
    "exit_conf": 0.72,
    "sampler_power": 0.65,
    "rarity_power": 0.68,
    "rarity_max_clip": 14.0,
    "weight_decay": 0.05,
    "lr": 0.0025,
    "rare_support_thresh": 90,
    "rare_max_mult": 3.0,
}


history = run_groq_driven_search(
    initial_params=seed_params,
    n_trials=5,
    best_artifact_path="best_loop_resnet_ensemble.pt",
    show_dashboard=False,
)

#after running training use the best model generated to generate a submission prediction
sub = inference(
    artifact_path="best_loop_resnet_ensemble.pt",
    submission_path="submission.csv",
)



########################################################################
STARTING GROQ-DRIVEN TRIAL 1/5
Reason : Baseline seed configuration.
Params : {
  "hidden_dim": 40,
  "dropout": 0.18,
  "input_noise_std": 0.05,
  "protected_noise_scale": 0.15,
  "residual_scale": 0.3,
  "exit_conf": 0.72,
  "sampler_power": 0.65,
  "rarity_power": 0.68,
  "rarity_max_clip": 14.0,
  "weight_decay": 0.05,
  "lr": 0.0025,
  "rare_support_thresh": 90,
  "rare_max_mult": 3.0
}
########################################################################

--- Fold 1/5 ---
F1E001 mF1:.144 b4:.010 PR:.242 L:1.950/1.154 AP[05 45 02 01 01 24 01 83 19 34 51]
F1E020 mF1:.335 b4:.038 PR:.371 L:.995/.789 AP[21 50 05 04 01 37 01 96 63 74 57]
F1E040 mF1:.370 b4:.022 PR:.393 L:.929/.724 AP[22 50 06 03 01 39 01 97 67 79 68]
F1E060 mF1:.378 b4:.062 PR:.408 L:1.013/.724 AP[22 55 06 12 01 36 00 96 71 80 70]
F1E080 mF1:.389 b4:.021 PR:.407 L:.838/.689 AP[23 55 04 04 01 36 00 97 72 81 74]
F1E100 mF1:.390 b4:.020 PR:.412 